# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD schema file for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print main metadata (title & description)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. This is important for referencing and extracting data using the `@id` of entities in the schema.

In [ ]:
# List all record sets by their @id and label
print("Record Sets in the dataset:")
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}  |  label: {record_set.label}")
    record_set_ids.append(record_set.id)

# For this dataset, print fields information for each record set
for record_set in dataset.record_sets:
    print(f"\nFields for Record Set '@id': {record_set.id}")
    for field in record_set.fields:
        print(f"  - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values discovered in the previous section.

In [ ]:
# Extract data from the main record set (assumed to be the first one for this dataset)

# You may want to choose or inspect the correct record set via its @id
print("Available record sets:")
for rid in record_set_ids:
    print(rid)

# If you know the main record set (from section 2), assign it here. For this dataset, often the first record set contains main protein data.
main_record_set_id = record_set_ids[0]  # adjust index if another is primary
print(f"\nLoading records from: {main_record_set_id}")

# Load ALL records for that record set
main_records = list(dataset.records(record_set=main_record_set_id))
main_df = pd.DataFrame(main_records)

print(f"Available columns for '@id' {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For demonstration, we'll select `MW` (Molecular Weight) as a numeric field and `description` as a group-by (if present). If not, modify fields as appropriate for the record set.

In [ ]:
# Choose field @id for numerical analysis (from earlier inspection)
# e.g., 'MW' for molecular weight, adjust to match actual column names present

# Try to pick a molecular weight or abundance numeric column
numeric_field_candidates = [col for col in main_df.columns if any(s in col.lower() for s in ['mw', 'molecular_weight', 'abundance', 'coverage', 'peptide', 'count', 'pi'])]

# Select the first one for the demonstration if any found
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    # fallback to a column likely to be numeric
    numeric_field = main_df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field for EDA: {numeric_field}")

threshold = main_df[numeric_field].quantile(0.5)  # median as threshold
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (median):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another categorical field if available (e.g., 'description', 'accession' etc.)
group_candidates = [col for col in main_df.columns if col != numeric_field and main_df[col].dtype == object]

if group_candidates:
    group_field = group_candidates[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("\nNo suitable group field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot after normalization (filtered)
plt.figure(figsize=(6, 4))
sns.boxplot(y=filtered_df[f"{numeric_field}_normalized"])
plt.title(f'Normalized {numeric_field} (records > median)')
plt.ylabel('Normalized Value')
plt.show()

# If group_field chosen, plot mean of numeric_field by group (if not too many groups)
if 'group_field' in locals() and grouped_df.shape[0] < 25:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

This exploration notebook demonstrates how to leverage the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library to load, inspect, and analyze datasets defined by [Croissant machine-readable metadata schemas](https://mlcommons.org/en/croissant/). We reviewed the FAIR^2 mass spectrometry protein dataset record sets, examined field structure, extracted data by referencing `@id` fields, and performed basic EDA and visualization. For further analysis, explore additional fields and relationships as defined in the metadata schema.
